# Notebook 08D — Final Comparative & Statistical Analysis

Version corrigée à partir du code fourni. Seule la gestion statistique des seeds dans le bootstrap a été modifiée : les prédictions stochastiques ne sont plus moyennées avant le calcul des métriques.


In [ ]:
# ============================================================
# NOTEBOOK 08D — FINAL COMPARATIVE & STATISTICAL ANALYSIS
# NO RETRAINING / NO MODEL SELECTION
#
# Combines frozen predictions from:
#   08A: Persistence, Ridge, RandomForest, XGBoost
#   08B: LSTM, TCN (3 seeds)
#   08C: GCN_native, GCN_operational, GCN_shuffled (3 seeds)
#
# Primary goals:
# 1) exact TEST alignment audit
# 2) final TEST comparison
# 3) per-device performance
# 4) performance by |ΔT15| regime
# 5) paired moving-block bootstrap CIs for ΔMAE and ΔRMSE vs Persistence
# 6) graph-topology paired bootstrap comparisons
#
# IMPORTANT:
# - No retraining.
# - No hyperparameter selection.
# - Stochastic models are summarized across the three predefined seeds.
# - Bootstrap resamples contiguous TIMESTAMP blocks, preserving all 7 devices
#   at each selected timestamp. This respects paired model comparisons and
#   partially accounts for temporal dependence.
# ============================================================

from pathlib import Path
import json, numpy as np, pandas as pd

ROOT = Path.cwd().resolve()
A = ROOT / "outputs" / "08A_tabular_models"
B = ROOT / "outputs" / "08B_lstm_tcn"
C = ROOT / "outputs" / "08C_gcn_graph_ablation"
OUT = ROOT / "outputs" / "08D_final_comparative_statistics"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_N = 13790
EXPECTED_TIMES = 1970
NODES = 7
TSEEDS = [42, 123, 2026]
BOOTSTRAP_REPS = 2000
BLOCK_LENGTH_TIMESTAMPS = 16   # 4 h at 15-min resolution; predeclared sensitivity choice
BOOTSTRAP_SEED = 20260915

for p in [
    A/"05_predictions_all_models.csv",
    B/"04_predictions_all_seeds.csv",
    C/"03_predictions_all_topologies_seeds.csv"
]:
    assert p.exists(), f"Missing required prediction file: {p}"

def norm_uuid(df):
    for c in ["uuid","device_id"]:
        if c in df.columns:
            if c != "uuid": df=df.rename(columns={c:"uuid"})
            return df
    return df

def norm_time(df):
    # Prefer target timestamp if available; otherwise prediction-origin timestamp.
    tc = "target_timestamp" if "target_timestamp" in df.columns else "timestamp"
    assert tc in df.columns, f"No timestamp column in {df.columns.tolist()}"
    df["eval_time"] = pd.to_datetime(df[tc], utc=True)
    return df


In [ ]:
# ---------------- LOAD TEST PREDICTIONS ----------------
a = pd.read_csv(A/"05_predictions_all_models.csv")
a = norm_time(norm_uuid(a))
a = a[a["split"].str.lower()=="test"].copy()
assert set(["model","y_true","y_pred"]).issubset(a.columns)
a["seed"] = -1
a["model_final"] = a["model"]

b = pd.read_csv(B/"04_predictions_all_seeds.csv")
b = norm_time(norm_uuid(b))
b = b[b["split"].str.lower()=="test"].copy()
b["model_final"] = b["model"]

c = pd.read_csv(C/"03_predictions_all_topologies_seeds.csv")

# 08C stores graph predictions by frozen node_index rather than UUID.
# Reconstruct the UUID identity using the exact 07D/07C frozen node ordering.
NODE_TO_UUID = {
    0: "99ca72a1-36de-4e79-b65d-16cdec7e42fb",
    1: "c6f4d193-bab8-438a-8706-d2f60e0ef81b",
    2: "d2948a1d-b2a0-435b-a9f6-abe67df61b70",
    3: "d9a2275e-351a-4095-9e35-7cae87c0112f",
    4: "f4a76a84-7b19-4f32-a681-3eabdaeaaabe",
    5: "f5c3c0e6-cf08-439f-8d31-fc739c100455",
    6: "fa54ef81-7df5-41e7-a5e8-5d97a69dc151",
}
assert "node_index" in c.columns, "08C predictions must contain node_index."
assert set(c["node_index"].dropna().astype(int).unique()) == set(NODE_TO_UUID), \
    f"Unexpected 08C node indices: {sorted(c['node_index'].dropna().unique())}"
c["uuid"] = c["node_index"].astype(int).map(NODE_TO_UUID)
assert c["uuid"].notna().all()

c = norm_time(c)
c = c[c["split"].str.lower()=="test"].copy()
c["model_final"] = c["topology"].map({
    "G_native":"GCN_native",
    "G_operational":"GCN_operational",
    "G_shuffled":"GCN_shuffled"
})
assert c["model_final"].notna().all()

KEEP = ["uuid","eval_time","model_final","seed","y_true","y_pred"]
allp = pd.concat([a[KEEP],b[KEEP],c[KEEP]],ignore_index=True)
allp["error"] = allp["y_pred"]-allp["y_true"]
allp["abs_error"] = allp["error"].abs()
allp["sq_error"] = allp["error"]**2


In [ ]:
# ---------------- ALIGNMENT AUDIT ----------------
reference = (
    allp[allp.model_final=="Persistence"][["uuid","eval_time","y_true"]]
    .drop_duplicates()
    .sort_values(["eval_time","uuid"])
    .reset_index(drop=True)
)
assert len(reference)==EXPECTED_N, len(reference)
assert reference["eval_time"].nunique()==EXPECTED_TIMES

audit=[]
for (model,seed),d in allp.groupby(["model_final","seed"]):
    q=d[["uuid","eval_time","y_true"]].sort_values(["eval_time","uuid"]).reset_index(drop=True)
    exact_keys = q[["uuid","eval_time"]].equals(reference[["uuid","eval_time"]])
    exact_y = len(q)==len(reference) and np.allclose(q["y_true"],reference["y_true"],rtol=0,atol=1e-6)
    audit.append({"model":model,"seed":seed,"n":len(q),
                  "unique_times":q["eval_time"].nunique(),
                  "exact_keys":exact_keys,"exact_y_true":exact_y})
audit=pd.DataFrame(audit)
assert audit["exact_keys"].all() and audit["exact_y_true"].all()
audit.to_csv(OUT/"01_exact_test_alignment_audit.csv",index=False)


In [ ]:
# ---------------- METRICS ----------------
def metric_row(y,p):
    y=np.asarray(y); p=np.asarray(p); e=p-y
    return {
        "n":len(y),
        "MAE":float(np.mean(np.abs(e))),
        "RMSE":float(np.sqrt(np.mean(e**2))),
        "R2":float(1-np.sum(e**2)/np.sum((y-y.mean())**2)),
        "Bias":float(np.mean(e)),
        "Error_SD":float(np.std(e,ddof=1))
    }

rows=[]
for (model,seed),d in allp.groupby(["model_final","seed"]):
    rows.append({"model":model,"seed":seed,**metric_row(d.y_true,d.y_pred)})
per_seed=pd.DataFrame(rows)
per_seed.to_csv(OUT/"02_TEST_metrics_per_model_seed.csv",index=False)

det_models=["Persistence","Ridge","RandomForest","XGBoost"]
stoch_models=["LSTM","TCN","GCN_native","GCN_operational","GCN_shuffled"]

final=[]
for model in det_models:
    r=per_seed[per_seed.model==model].iloc[0]
    final.append({"model":model,"type":"deterministic",
                  "n":int(r.n),"MAE_mean":r.MAE,"MAE_SD_seeds":np.nan,
                  "RMSE_mean":r.RMSE,"RMSE_SD_seeds":np.nan,
                  "R2_mean":r.R2,"R2_SD_seeds":np.nan,
                  "Bias_mean":r.Bias,"Bias_SD_seeds":np.nan})
for model in stoch_models:
    d=per_seed[per_seed.model==model]
    final.append({"model":model,"type":"3-seed mean",
                  "n":EXPECTED_N,
                  "MAE_mean":d.MAE.mean(),"MAE_SD_seeds":d.MAE.std(ddof=1),
                  "RMSE_mean":d.RMSE.mean(),"RMSE_SD_seeds":d.RMSE.std(ddof=1),
                  "R2_mean":d.R2.mean(),"R2_SD_seeds":d.R2.std(ddof=1),
                  "Bias_mean":d.Bias.mean(),"Bias_SD_seeds":d.Bias.std(ddof=1)})
final=pd.DataFrame(final).sort_values("RMSE_mean")
pers_rmse=float(final.loc[final.model=="Persistence","RMSE_mean"].iloc[0])
final["RMSE_change_vs_persistence_pct"]=100*(final.RMSE_mean-pers_rmse)/pers_rmse
final.to_csv(OUT/"03_FINAL_TEST_model_comparison.csv",index=False)


In [ ]:
# ---------------- PER DEVICE ----------------
dev=[]
for (model,seed,uuid),d in allp.groupby(["model_final","seed","uuid"]):
    dev.append({"model":model,"seed":seed,"uuid":uuid,**metric_row(d.y_true,d.y_pred)})
dev=pd.DataFrame(dev)
dev.to_csv(OUT/"04_TEST_per_device_per_seed.csv",index=False)

devsum=(dev.groupby(["model","uuid"])
        .agg(MAE_mean=("MAE","mean"),MAE_SD_seeds=("MAE","std"),
             RMSE_mean=("RMSE","mean"),RMSE_SD_seeds=("RMSE","std"),
             R2_mean=("R2","mean"),Bias_mean=("Bias","mean"))
        .reset_index())
devsum.to_csv(OUT/"05_TEST_per_device_summary.csv",index=False)


In [ ]:
# ---------------- TEMPERATURE-CHANGE REGIMES ----------------
pers = allp[allp.model_final=="Persistence"][["uuid","eval_time","y_true","y_pred"]].copy()
pers["abs_delta_T15"]=(pers["y_true"]-pers["y_pred"]).abs()
def bin_change(x):
    if x < .02: return "<0.02"
    if x < .05: return "0.02-0.05"
    if x < .10: return "0.05-0.10"
    if x < .20: return "0.10-0.20"
    return ">=0.20"
pers["change_regime"]=pers["abs_delta_T15"].map(bin_change)
allr=allp.merge(pers[["uuid","eval_time","abs_delta_T15","change_regime"]],
                on=["uuid","eval_time"],how="left",validate="many_to_one")
reg=[]
for (model,seed,rg),d in allr.groupby(["model_final","seed","change_regime"]):
    reg.append({"model":model,"seed":seed,"change_regime":rg,
                "n":len(d),**metric_row(d.y_true,d.y_pred)})
reg=pd.DataFrame(reg)
reg.to_csv(OUT/"06_TEST_metrics_by_temperature_change_per_seed.csv",index=False)
regsum=(reg.groupby(["model","change_regime"])
        .agg(n=("n","max"),MAE_mean=("MAE","mean"),MAE_SD_seeds=("MAE","std"),
             RMSE_mean=("RMSE","mean"),RMSE_SD_seeds=("RMSE","std"),
             Bias_mean=("Bias","mean"))
        .reset_index())
regsum.to_csv(OUT/"07_TEST_metrics_by_temperature_change_summary.csv",index=False)


In [ ]:
# ---------------- SEED-AWARE PREDICTIONS FOR STOCHASTIC MODELS ----------------
# Keep the three predefined training seeds separate.
# Metrics are computed seed-by-seed and then averaged at metric level.
# This preserves the same estimand used in the final TEST comparison.

models=final["model"].tolist()
times=np.sort(reference.eval_time.unique())
uuids=np.sort(reference.uuid.unique())
assert len(times)==EXPECTED_TIMES and len(uuids)==NODES

# Common y_true matrix [time, node]
yy=(reference.pivot(index="eval_time",columns="uuid",values="y_true")
    .reindex(index=times,columns=uuids))
assert yy.notna().all().all()
Y=yy.to_numpy()

# P[model] is:
# - one [time,node] matrix for deterministic models
# - a dict {seed: [time,node] matrix} for stochastic models
P={}

for model in det_models:
    d=allp[allp.model_final==model]
    assert d["seed"].nunique()==1
    py=(d.pivot(index="eval_time",columns="uuid",values="y_pred")
        .reindex(index=times,columns=uuids))
    yt=(d.pivot(index="eval_time",columns="uuid",values="y_true")
        .reindex(index=times,columns=uuids))
    assert py.notna().all().all() and yt.notna().all().all()
    assert np.allclose(Y,yt.to_numpy(),atol=1e-6)
    P[model]=py.to_numpy()

for model in stoch_models:
    P[model]={}
    dmodel=allp[allp.model_final==model]
    observed_seeds=sorted(dmodel["seed"].astype(int).unique())
    assert observed_seeds==sorted(TSEEDS), (model,observed_seeds,TSEEDS)

    for seed in TSEEDS:
        d=dmodel[dmodel["seed"].astype(int)==int(seed)]
        py=(d.pivot(index="eval_time",columns="uuid",values="y_pred")
            .reindex(index=times,columns=uuids))
        yt=(d.pivot(index="eval_time",columns="uuid",values="y_true")
            .reindex(index=times,columns=uuids))
        assert py.notna().all().all() and yt.notna().all().all()
        assert np.allclose(Y,yt.to_numpy(),atol=1e-6)
        P[model][int(seed)]=py.to_numpy()


In [ ]:
# ---------------- MOVING-BLOCK BOOTSTRAP ----------------
rng=np.random.default_rng(BOOTSTRAP_SEED)
T=len(times)
L=BLOCK_LENGTH_TIMESTAMPS
nblocks=int(np.ceil(T/L))

# Generate the paired bootstrap resamples once and reuse them
# for every comparison.
bootstrap_indices=[]
for _ in range(BOOTSTRAP_REPS):
    starts=rng.integers(0,T-L+1,size=nblocks)
    idx=np.concatenate([np.arange(s,s+L) for s in starts])[:T]
    bootstrap_indices.append(idx)

def loss_metrics(y,p):
    e=p-y
    return np.mean(np.abs(e)), np.sqrt(np.mean(e**2))

def model_metrics(model,idx=None):
    """Return metrics using the same estimand as the final table."""
    yy=Y if idx is None else Y[idx]

    if model in det_models:
        pp=P[model] if idx is None else P[model][idx]
        return loss_metrics(yy,pp)

    # Stochastic models: compute metric per seed, then average metrics.
    maes=[]; rmses=[]
    for seed in TSEEDS:
        pp=P[model][int(seed)]
        if idx is not None:
            pp=pp[idx]
        ma,rm=loss_metrics(yy,pp)
        maes.append(ma); rmses.append(rm)

    return float(np.mean(maes)),float(np.mean(rmses))

# Verify that the corrected bootstrap estimand reproduces
# exactly the final TEST metrics already reported.
for model in models:
    ma,rm=model_metrics(model)
    expected_ma=float(final.loc[final.model==model,"MAE_mean"].iloc[0])
    expected_rm=float(final.loc[final.model==model,"RMSE_mean"].iloc[0])
    assert np.isclose(ma,expected_ma,atol=1e-8), (model,ma,expected_ma)
    assert np.isclose(rm,expected_rm,atol=1e-8), (model,rm,expected_rm)

print("✓ Observed bootstrap estimand exactly reproduces final metrics")

def bootstrap_comparison(first,second):
    oma,orm=model_metrics(first)
    bma,brm=model_metrics(second)

    vm=[]; vr=[]
    for idx in bootstrap_indices:
        ma,rm=model_metrics(first,idx)
        ba,br=model_metrics(second,idx)
        vm.append(ma-ba)
        vr.append(rm-br)

    return {
        "comparison":f"{first} - {second}",
        "delta_MAE_observed":oma-bma,
        "delta_MAE_CI95_low":np.quantile(vm,.025),
        "delta_MAE_CI95_high":np.quantile(vm,.975),
        "delta_RMSE_observed":orm-brm,
        "delta_RMSE_CI95_low":np.quantile(vr,.025),
        "delta_RMSE_CI95_high":np.quantile(vr,.975)
    }

comparisons=[]

# Primary: every model vs Persistence.
for model in models:
    if model=="Persistence":
        continue
    comparisons.append(bootstrap_comparison(model,"Persistence"))

# Graph structural comparisons.
for first,second in [
    ("GCN_operational","GCN_native"),
    ("GCN_shuffled","GCN_operational"),
    ("GCN_shuffled","GCN_native")
]:
    comparisons.append(bootstrap_comparison(first,second))

boot=pd.DataFrame(comparisons)
boot.to_csv(OUT/"08_paired_moving_block_bootstrap_CI.csv",index=False)


In [ ]:
# ---------------- COMPACT REPORT ----------------
manifest={
    "analysis_only":True,
    "no_retraining":True,
    "no_hyperparameter_retuning":True,
    "test_n_node_level":EXPECTED_N,
    "test_unique_timestamps":EXPECTED_TIMES,
    "devices":NODES,
    "stochastic_prediction_handling":
        "training seeds retained separately; metrics computed per seed and averaged at metric level within each bootstrap resample",
    "prediction_averaging_before_metric":False,
    "bootstrap":"paired moving-block bootstrap over timestamps; all 7 devices retained per sampled timestamp",
    "bootstrap_repetitions":BOOTSTRAP_REPS,
    "block_length_timestamps":BLOCK_LENGTH_TIMESTAMPS,
    "block_length_minutes":BLOCK_LENGTH_TIMESTAMPS*15,
    "bootstrap_seed":BOOTSTRAP_SEED,
    "stochastic_training_seeds":TSEEDS,
    "bootstrap_estimand":
        "difference between mean-across-training-seed metrics for stochastic models and the comparator metric",
    "note":"Block length is a predeclared pragmatic 4-hour choice; no retraining or post-hoc hyperparameter selection."
}
with open(OUT/"09_statistical_analysis_manifest.json","w") as f:
    json.dump(manifest,f,indent=2)

print("="*90)
print("08D — FINAL COMPARATIVE & STATISTICAL ANALYSIS")
print("="*90)
print("\nFINAL TEST RANKING")
print(final[["model","type","MAE_mean","MAE_SD_seeds","RMSE_mean","RMSE_SD_seeds",
             "R2_mean","Bias_mean","RMSE_change_vs_persistence_pct"]].to_string(index=False))

print("\nPAIRED MOVING-BLOCK BOOTSTRAP — 95% CI")
print(boot.to_string(index=False))

print("\nVALIDATION")
print("✓ No model retraining")
print("✓ Exact 13,790 TEST node-level observations aligned across every model/seed")
print("✓ Exact 1,970 TEST timestamps × 7 devices")
print("✓ Stochastic models summarized across predefined seeds")
print("✓ Stochastic seeds retained separately during bootstrap")
print("✓ Metrics computed seed-by-seed and averaged at metric level")
print("✓ No averaging of stochastic predictions before metric calculation")
print("✓ Paired contiguous timestamp blocks preserve all devices")
print("✓ Per-device and |ΔT15| regime analyses saved")
print("✓ Graph topology comparisons included")
